# attenzione: fare esercizzi lezione 6

# discesa gradiente stocastica
**discesa gradiente**:
* per calcolare $\nabla J(w)$ usa l'intero dataset, ossia la matrice $X$
* se $X$ enorme, non e' possibile usare la discesa del gradiente.

**discesa gradiente stocastica**: divido $X$ in blocchi, ciascuno con un numero fissato di righe, possibilmente ogni blocco le sceglie randomicamente da $X$.
* $X_b$ e' la matrice $b$ esima, ottenuta.
* **a che serve**: non calcolo il gradiente su $X$, ma piuttosto su un unico $X_i$
* **stocastica**: voglio che ogni blocco abbia righe scelte in modo uniforme a random, in modo che un blocco sia rappresentativo dell'intero data set (gemini conferma)

$$
\nabla J(w) = - X^T_b \times \text{errors}
$$

**pro dell'approccio stocastico**:
* moltiplicazioni con matrici piccole
* vedremo che converge prima


Come si implementa questo modello? In `fit`
* **shuffle**: i campioni sono mescolati ad ogni iterazione
* il dataset e' suddiviso in blocchi di dimensione fissata
* per ogni blocco, calcolo il gradiente su quello
* a fine iterazione calcola e registra il costo sull'intero dataset, devo monitorare la convergenza


In [ ]:
import matplotlib.pyplot as pyplot
import numpy as np

class LinearRegressionSGD(object):
    def __init__(self, eta=0.001, n_iter=1000, tol=1e-5, batch_size=None):
        self.eta = eta
        self.n_iter = n_iter
        self.costi_ = []
        self.tol_ = tol
        self.batch_size_ = batch_size
    
    def fit(self, X, y):
        self.batch_size_ = X.shape[0] if self.batch_size_ == None else self.batch_size_

        # Standardizzazione delle feature
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)

        X_std = (X - self.mean_) / self.std_
        
        # Inizializzazione pesi
        self.w_ = np.zeros(1 + X.shape[1])

        for _ in range(self.n_iter):
            # crea e permuta gli indici
            shuffle_idxs = np.arange(X.shape[0])
            np.random.shuffle(shuffle_idxs)

            # itera a step di batch_size
            for b_idx in range(0, X.shape[0], self.batch_size_):
                # prendi i valori corrispondenti al blocco
                idxs_b = shuffle_idxs[b_idx:b_idx+self.batch_size_]
                X_b = X_std[idxs_b]
                y_b = y[idxs_b]
                
                # calcola la funzione in X_b e calcola gli errori9
                output = self.net_input(X_b)
                errors = y_b - output

                # aggiorna i pesi
                self.w_[1:] += self.eta * X_b.T.dot(errors)/self.batch_size_
                self.w_[0] += self.eta * errors.sum()/self.batch_size_
            
            # ora fai la predizione su tutte le righe e vedi quant'e' l'errore
            output = self.net_input(X_std)
            self.costi_.append(mae(y,output))
            if len(self.costi_) >= 2:
                if np.abs(self.costi_[-1]-self.costi_[-2]) < self.tol_: break

    # il resto non cambia!
    def net_input(self, X):
        return np.dot(X, self.w_[1:]) + self.w_[0]
        
    def predict(self, X):
        # Standardizza i dati usando media e std del training set
        X_std = (X - self.mean_) / self.std_
        
        return self.net_input(X_std)
    
    def get_coef(self):
        """
        Restituisce i coefficienti e l'intercetta del modello
        riferiti ai dati originali (non standardizzati).
        """

        # Coefficienti originali
        coef = self.w_[1:] / self.std_
        # Intercetta originale
        intercept = self.w_[0] - np.sum(self.w_[1:] * self.mean_ / self.std_)
        
        return intercept, coef

def mae(y, z):
    '''
    Calcola l'errore medio assoluto
    '''
    return np.sum(np.abs(y-z))/y.shape[0]

# sperimentiamo con `sklearn`
`make_regression`: crea un problema di regressione lineare. con dei parametri otteniamo $X,y$ per testare il nostro modello.

In [ ]:
from sklearn.datasets import make_regression

# crea un 
X, y = make_regression(
    n_samples=2000,
    n_features=10,
    noise=10.0,         # rumore da aggiugnere al target
    random_state=42     # fisso il seed per il generatore di numeri casuale
)

linregSGD = LinearRegressionSGD(batch_size=500, n_iter=10000, tol=0.000001)
linregSGD.fit(X,y)
errorSGD = mae(y, linregSGD.predict(X))

print(f"Err: {errorSGF:.2f}, Iter: {len(linregSGD.costi_)}")



# considerazioni
* **eta troppo piccolo**: rischio di fare overfitting sul training set.
* **in generale**: c'e' sempre rischio di overfitting
* **soluzione**: introduco il ***validation set***

Man mano che addestro sul training set, vedo che succede sul validation set.
* **caso 1**: training set ovviamento migliora, e le performance sul validation anche
* **caso 2**: le performance sul validation set peggiorano, sto facendo overfitting (mi adatto al rumore sul campione).

# implementazione
qui sotto implementiamo un fit che tiene conto anche del **validation set**: dopo una iterazione di SGD, testo il validation set:
* se cambia rispetto all'ultima iterazione di un fattore $tol$, allora lo aggiorno
* altrimenti a meno che non supero `patience` faccio un'altra iterazione

In [ ]:
import matplotlib.pyplot as pyplot
import numpy as np

class LinearRegressionSGD(object):
    def __init__(self, eta=0.001, n_iter=1000, tol=1e-5, batch_size=None):
        self.eta = eta
        self.n_iter = n_iter
        self.costi_ = []
        self.tol_ = tol
        self.batch_size_ = batch_size
    
    def fit(self, X, y, X_val=None, y_val=None, patience=20):
        self.batch_size_ = X.shape[0] if self.batch_size_ == None else self.batch_size_

        # Standardizzazione delle feature
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)

        X_std = (X - self.mean_) / self.std_

        # standardizza validation set 
        if X_val is not None:
            X_val_std = (X_val - self.mean_) / self.std_
        
        # Inizializzazione pesi
        self.w_ = np.zeros(1 + X.shape[1])

        # validation variables
        best_val_loss = np.inf
        best_W = self.w_.copy()
        patience_count = 0

        for _ in range(self.n_iter):
            # crea e permuta gli indici
            shuffle_idxs = np.arange(X.shape[0])
            np.random.shuffle(shuffle_idxs)

            # itera a step di batch_size
            for b_idx in range(0, X.shape[0], self.batch_size_):
                # prendi i valori corrispondenti al blocco
                idxs_b = shuffle_idxs[b_idx:b_idx+self.batch_size_]
                X_b = X_std[idxs_b]
                y_b = y[idxs_b]
                
                # calcola la funzione in X_b e calcola gli errori9
                output = self.net_input(X_b)
                errors = y_b - output

                # aggiorna i pesi
                self.w_[1:] += self.eta * X_b.T.dot(errors)/self.batch_size_
                self.w_[0] += self.eta * errors.sum()/self.batch_size_
            
            # ora fai la predizione su tutte le righe e vedi quant'e' l'errore
            output = self.net_input(X_std)
            train_loss = mae(y,output)
            self.costi_.append(train_loss)

            # QUI AGGIUNGIAMO LA VALIDAZIONE RISPETTO A PRIMA
            if X_val is not None:
                # calcola loss sul validation set 
                val_output = self.net_input(X_val_std)
                val_loss = mae(y_val, val_output)

                if val_loss < best_val_loss - self.tol_:
                    best_val_loss = val_loss
                    best_w = self.w_.copy()
                    patience_count = 0
                else: 
                    patience_count += 1

                if patience_count >= patience:
                    print(f"interrupted after {epoca} steps")
                    break

            # la vecchia condizione di arresto
            elif len(self.costi_) >= 2:
                if np.abs(self.costi_[-1]-self.costi_[-2]) < self.tol_: break

    # il resto non cambia!
    def net_input(self, X):
        return np.dot(X, self.w_[1:]) + self.w_[0]
        
    def predict(self, X):
        # Standardizza i dati usando media e std del training set
        X_std = (X - self.mean_) / self.std_
        
        return self.net_input(X_std)
    
    def get_coef(self):
        """
        Restituisce i coefficienti e l'intercetta del modello
        riferiti ai dati originali (non standardizzati).
        """

        # Coefficienti originali
        coef = self.w_[1:] / self.std_
        # Intercetta originale
        intercept = self.w_[0] - np.sum(self.w_[1:] * self.mean_ / self.std_)
        
        return intercept, coef

In [ ]:
X, y = make_regression(
    n_samples=2000,       # numero di campioni
    n_features=100,        # numero di feature
    noise=10.0,          # rumore da aggiungere ai target
    random_state=42      # per riproducibilità
)

n = X.shape[0]
train_size = 0.6
val_size = 0.2
np.random.seed(10)
train_idxs = np.random.choice(n, size = int(n*train_size), replace=False)

# fai lo split di questo in validation e test
test_val_idxs =  np.setdiff1d(np.arange(n), train_idxs)

val_idxs = np.random.choice(test_val_idxs, size = int(n*val_size), replace=False)
test_idxs = np.setdiff1d(test_val_idxs, val_idxs)

X_train, X_val, X_test = X[train_idxs], X[val_idxs], X[test_idxs]
y_train, y_val, y_test = y[train_idxs], y[val_idxs], y[test_idxs]

linregSGD = LinearRegressionSGD(batch_size=100, n_iter=10000, tol=0.001)
linregSGD.fit(X_train, y_train, X_val=X_val, y_val=y_val, patience=5)

errorSGD = mae(y_train, linregSGD.predict(X_train))
print(f"Mean Absolute Error (MAE) train set: {errorSGD:.2f}. Iterazioni: {len(linregSGD.costi_)}")

errorSGD = mae(y_test, linregSGD.predict(X_test))
print(f"Mean Absolute Error (MAE) test set: {errorSGD:.2f}. Iterazioni: {len(linregSGD.costi_)}")

In questo modello, in media, il mae sul train test e sul test set sono molto vicini. ovviamente mi aspetto che `mae_test_set >> mae_train_set` (ossia piu errori sul test set che sul train set)

**overfitting**: con `scikit` genero 2000 sample da 100 feature non va bene! Ho troppe feature e pochi campioni, dunque il fenomeno non si distribuisce bene sui sample e dunque c'e' rischio di overfitting